# Vorgehen

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation




# 1. Business Understanding

Grundlagen des Datensatzes:
- 50.000 Essens-Bestellungen
- 6 indische Städte, Beträge in INR
- Merkmale
    - Bestellkontext: Uhrzeit, Wochenende/Werktag, Regen
    - Bestellung: Küche, Mahlzeit, Restauranttyp, Bestellwert, Rabatt, Liefergebühr
    - Verhalten: Bewertung, Zeit Wiederholungsbestellung
    - Situation: Stimmung, Hunger, Begleichtung
    - Alter

Background:
- Bisher werden Rabatte vermutlich an jeden gegeben. Im Datensatz bekommt ungefähr jede zweite Bestellung einen Rabatt.

Anwendungsfall:
- Vorhersage von Wiederholungsbestellungen für gezielte Rabattvergabe (Auf aktuell 50.000 Bestellungen 4.000 Nutzer)
    - Zielvariable: is_repeat_order
    - Nach jeder Bestellung schätzt das Modell, wie wahrscheinlich der Kunde wieder bestellt, je nachdem wird ein Rabatt vergeben
        - Hohe Wahrscheinlichkeit -> kein Rabatt
        - Geringe Wahrscheinlichkeit -> Rabatt


# 2. Data Understanding

Datensatz:
- https://www.kaggle.com/datasets/rhythmghai/food-ordering-behavior-india-50k-orders?resource=download
- Synthetische Daten
- Autor: Rhythm_Ghai

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("food_ordering_behavior_dataset.csv")
df.head()

,order_id,user_id,age,city,order_time,day_type,cuisine,meal_type,restaurant_type,order_value,discount_applied,delivery_fee,time_taken_to_order,rating_given,is_repeat_order,mood,hunger_level,company,rainy_weather
0,1,2698,35,Pune,Evening,Weekend,Chinese,Dinner,Premium,971,Yes,90,13,1,Yes,Celebrating,High,Partner,No
1,2,3237,44,Mumbai,Night,Weekend,South Indian,Dinner,Budget,442,No,26,13,2,No,Lazy,Low,Family,No
2,3,3626,31,Delhi,Morning,Weekend,Biryani,Breakfast,Mid-range,739,Yes,85,10,2,Yes,Happy,Medium,Friends,No
3,4,3176,23,Delhi,Evening,Weekend,Biryani,Snacks,Mid-range,466,No,44,12,2,No,Happy,Medium,Alone,No
4,5,4824,26,Chandigarh,Morning,Weekday,Chinese,Lunch,Premium,927,Yes,58,13,2,No,Happy,Medium,Partner,Yes


### Fehlende Werte und Duplikate

Da es sich um synthetisch generierte und bereits qualitätsgesicherte Daten handelt, sind fehlende Werte und Duplikate erwartungsgemäß nicht vorhanden. Die Prüfung bestätigt dies. Eine weiterführende Untersuchung der Datenqualität ist an dieser Stelle daher nicht notwendig.

In [3]:
print("Fehlende Werte pro Spalte:")
print(df.isna().sum())
print("Duplizierte Zeilen:", df.duplicated().sum())
print("Duplizierte order_id:", df["order_id"].duplicated().sum())

Fehlende Werte pro Spalte:
order_id               0
user_id                0
age                    0
city                   0
order_time             0
day_type               0
cuisine                0
meal_type              0
restaurant_type        0
order_value            0
discount_applied       0
delivery_fee           0
time_taken_to_order    0
rating_given           0
is_repeat_order        0
mood                   0
hunger_level           0
company                0
rainy_weather          0
dtype: int64
Duplizierte Zeilen: 0
Duplizierte order_id: 0


## 3 - Data Preparation

In [ ]:
target = "is_repeat_order"
numeric_cols = ["age", "order_value", "delivery_fee", "time_taken_to_order", "rating_given"]
categorical_cols = [
    "city", "order_time", "day_type", "cuisine", "meal_type", "restaurant_type",
    "discount_applied", "mood", "hunger_level", "company", "rainy_weather",
]

df[target].value_counts(normalize=True)

Die Zielgröße ist nahezu perfekt ausgeglichen. Die Accuracy ist daher eine sinnvolle Metrik, und ein Modell, das immer die häufigste Klasse rät, käme auf etwa 50%.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(data=df, x=col, hue=target, bins=30, element="step", ax=ax)
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, col in zip(axes.ravel(), categorical_cols):
    rate = df.groupby(col)[target].apply(lambda s: (s == "Yes").mean()).sort_values()
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="steelblue")
    ax.axvline(0.5, color="red", linestyle="--")
    ax.set_xlim(0.4, 0.6)
    ax.set_title(f"Anteil Wiederholungsbestellungen nach {col}")
    ax.set_xlabel("")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
corr_df = df[numeric_cols].copy()
corr_df[target] = (df[target] == "Yes").astype(int)
corr_df["discount_applied"] = (df["discount_applied"] == "Yes").astype(int)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Korrelationen")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x="restaurant_type", y="order_value", ax=axes[0])
sns.countplot(data=df, x="meal_type", hue="order_time", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
orders_per_user = df.groupby("user_id").size()
print("Nutzer:", orders_per_user.shape[0])
print(orders_per_user.describe())
print("Nutzer mit genau einem Wohnort:", (df.groupby("user_id")["city"].nunique() == 1).mean())
print("Nutzer mit genau einem Alter:", (df.groupby("user_id")["age"].nunique() == 1).mean())

### Erkenntnisse aus der EDA
- Es gibt keine fehlenden Werte und keine Duplikate.
- Zielgröße und alle kategorialen Merkmale sind nahezu gleichverteilt, `order_value` ist über 100 bis 999 gleichverteilt, es gibt also kaum Ausreißer.
- Die Anteile der Wiederholungsbestellungen liegen in allen Gruppen sehr nah an 50% und die Korrelationen sind praktisch 0. Der Datensatz sieht stark nach zufällig (synthetisch) erzeugten Daten aus. Wir erwarten deshalb, dass kein Modell deutlich besser als Raten wird.
- Die 50.000 Bestellungen gehören zu nur 4.000 Nutzern. Die Bestellungen desselben Nutzers dürfen beim Split nicht auf Trainings- und Testdaten verteilt werden (Datenleck).

## 3 - Feature Engineering und Train/Test-Split

In [ ]:
features = df.drop(columns=["order_id", "user_id", target]).copy()

# Zusätzliche Merkmale
features["fee_ratio"] = features["delivery_fee"] / features["order_value"]
features["total_cost"] = features["order_value"] + features["delivery_fee"]
features["is_evening_or_night"] = features["order_time"].isin(["Evening", "Night"]).astype(int)

labels = (df[target] == "Yes").astype(int)

numeric_features = numeric_cols + ["fee_ratio", "total_cost", "is_evening_or_night"]
categorical_features = categorical_cols
features.head()

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Split nach Nutzer, damit kein Nutzer in Training und Test vorkommt
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(features, labels, groups=df["user_id"]))

train_data, test_data = features.iloc[train_idx], features.iloc[test_idx]
train_label, test_label = labels.iloc[train_idx], labels.iloc[test_idx]
train_groups = df["user_id"].iloc[train_idx]

print("Training:", train_data.shape, "Test:", test_data.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preparation_pipeline = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

## 4 - Modelling

Als Vergleichsbasis dient ein `DummyClassifier`. Danach trainieren wir eine logistische Regression und einen Random Forest. Die Hyperparameter werden mit `GridSearchCV` und einer gruppierten Kreuzvalidierung (`GroupKFold`) gewählt.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline

cv = GroupKFold(n_splits=5)

candidates = {
    "Dummy (häufigste Klasse)": (
        Pipeline([("preparation", preparation_pipeline), ("model", DummyClassifier(strategy="most_frequent"))]),
        {},
    ),
    "Logistische Regression": (
        Pipeline([("preparation", preparation_pipeline), ("model", LogisticRegression(max_iter=1000))]),
        {"model__C": [0.01, 0.1, 1, 10]},
    ),
    "Random Forest": (
        Pipeline([("preparation", preparation_pipeline), ("model", RandomForestClassifier(random_state=0, n_jobs=-1))]),
        {"model__n_estimators": [100, 200], "model__max_depth": [4, 8]},
    ),
}

results = {}
for name, (pipeline, param_grid) in candidates.items():
    search = GridSearchCV(pipeline, param_grid, cv=cv, scoring="accuracy")
    search.fit(train_data, train_label, groups=train_groups)
    results[name] = search
    print(f"{name}: CV-Accuracy = {search.best_score_:.4f}, beste Parameter = {search.best_params_}")

## 5 - Evaluation/Deployment

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

rows = []
for name, search in results.items():
    pred = search.predict(test_data)
    proba = search.predict_proba(test_data)[:, 1]
    rows.append({
        "Modell": name,
        "Accuracy (Train)": search.score(train_data, train_label),
        "Accuracy (Test)": accuracy_score(test_label, pred),
        "ROC-AUC (Test)": roc_auc_score(test_label, proba),
    })

pd.DataFrame(rows).set_index("Modell").round(4)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

best_name = max(
    (n for n in results if not n.startswith("Dummy")),
    key=lambda n: results[n].best_score_,
)
best_model = results[best_name]
print("Bestes Modell (nach CV):", best_name)
print(classification_report(test_label, best_model.predict(test_data), target_names=["Nein", "Ja"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(best_model, test_data, test_label, display_labels=["Nein", "Ja"], ax=axes[0])
RocCurveDisplay.from_estimator(best_model, test_data, test_label, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    best_model, test_data, test_label, n_repeats=5, random_state=0, scoring="accuracy"
)
importance_df = (
    pd.Series(importance.importances_mean, index=test_data.columns)
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance_df.values, y=importance_df.index, color="steelblue")
plt.title("Permutation Importance (Test, Abnahme der Accuracy)")
plt.show()

### Interpretation
__Ihre Antwort:__ Bitte nach dem Ausführen anhand der Ergebnisse ergänzen bzw. prüfen.

Erwartetes Ergebnis: Alle Modelle liegen bei einer Accuracy und einem ROC-AUC nahe an den Werten des Dummy-Modells (etwa 0,50). Die Permutation Importance ist überall nahe 0. Das passt zur EDA: In den Daten gibt es keine erkennbaren Zusammenhänge zwischen den Merkmalen und der Wiederholungsbestellung. Der Datensatz enthält vermutlich zufällig erzeugte Werte.

Konsequenzen:
- Mit diesen Merkmalen lässt sich nicht vorhersagen, ob eine Bestellung eine Wiederholungsbestellung ist. Das Modell ist praktisch nicht einsetzbar.
- Wichtig ist der Vergleich mit dem Dummy-Modell. Eine Accuracy von 50% wirkt nur auf den ersten Blick schlecht, sie ist bei ausgeglichenen Klassen der Zufallswert.
- Der Split nach Nutzer verhindert ein Datenleck. Bei einem zufälligen Split könnten Nutzerspezifika zu scheinbar besseren Ergebnissen führen.
- Sinnvolle nächste Schritte wären echte Verhaltensdaten, zum Beispiel Bestellhistorie pro Nutzer (Abstände zwischen Bestellungen, Anzahl bisheriger Bestellungen).